# 分類性能と公平性 — ResNet vs attribute-invariant ResNet（3 seed）

通常 ResNet と attribute-invariant ResNet を、**seed 43 / 44 / 45 の 3 本ずつ**で比べる。
どちらも best validation-AUROC checkpoint を、同じ test split で評価する。
invariant 化で分類性能をどれだけ払い、その見返りに属性間の差がどう動いたかを見る。

**n=3 なので、平均だけでは読めない。** 手法ごとの平均±SD（標本 SD、自由度 2）を棒とひげで出し、
**3 seed の値そのものを点で重ねる**。ひげが重なるかどうかと、3 点がどう散っているかを同時に見る。
SD は揺れの目安であって検定ではない。

seed 42 は混ぜない。s42 の invariant は adversary が全属性にかかっており、ここで読む 3 本
（sex / race / age のみ）とは別の手法になる。

集計は `groups.py` が持つ。この notebook は `results/*.csv` だけを読み、表示・作図・読み取りを行う。

```bash
uv run python analysis/common/predictions.py --study initial_resnet_vs_invariant --split test \
  --run-dir projects/hypernet_e2e/runs/20260922T063725Z-resnet-chexpert-s43-efcd \
  --run-dir projects/hypernet_e2e/runs/20260922T063727Z-resnet-chexpert-s44-6f0c \
  --run-dir projects/hypernet_e2e/runs/20260922T063727Z-resnet-chexpert-s45-fb77 \
  --run-dir projects/hypernet_e2e/runs/20260922T063725Z-resnet-chexpert-attribute-invariant-s43-546c \
  --run-dir projects/hypernet_e2e/runs/20260922T063728Z-resnet-chexpert-attribute-invariant-s44-8e6b \
  --run-dir projects/hypernet_e2e/runs/20260922T063728Z-resnet-chexpert-attribute-invariant-s45-404a
uv run python analysis/initial_resnet_vs_invariant/groups.py --split test
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.style
import numpy as np
import pandas as pd
import rootutils

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from analysis.common.paths import STYLE_SHEET  # noqa: E402

matplotlib.style.use(STYLE_SHEET)

RESULTS = ROOT / "analysis" / "initial_resnet_vs_invariant" / "results"
SPLIT = "test"
RESNET, INVARIANT = "ResNet", "attribute-invariant ResNet"
# 色は手法を表す。2 条件しか無いので style sheet の Okabe-Ito の先頭 2 色を割り当てる。
MODEL_COLOR = {RESNET: "#0173B2", INVARIANT: "#DE8F05"}
SEED_MARK = {"color": "#2b2b2b", "s": 13, "zorder": 4, "linewidths": 0}

performance = pd.read_csv(RESULTS / f"classification_performance_{SPLIT}.csv")
performance_by_method = pd.read_csv(RESULTS / f"classification_performance_by_method_{SPLIT}.csv")
fairness = pd.read_csv(RESULTS / f"fairness_metrics_{SPLIT}.csv")
fairness_by_method = pd.read_csv(RESULTS / f"fairness_metrics_by_method_{SPLIT}.csv")
ATTRIBUTES = list(dict.fromkeys(fairness["attribute"]))
SEEDS = sorted(performance["seed"].unique())

## 描画の共通部分

3 つの図はどれも「category ごとに手法別の棒を並べ、平均±SD のひげと seed の点を重ねる」形なので、
描画を 1 つと、表から値を取り出す小さな関数を 2 つ持つ。再利用も一括出力もしないので
`plots.py` には出さない（`analysis/AGENTS.md`）。

In [ ]:
def method_stats(summary, metric, attributes=None):
    """集約表から、category 順の平均と SD を手法別に取り出す。"""
    rows = summary[summary["metric"] == metric]
    mean, std = {}, {}
    for model in MODEL_COLOR:
        part = rows[rows["model"] == model]
        if attributes is None:
            mean[model], std[model] = [part["mean"].iloc[0]], [part["std"].iloc[0]]
            continue
        indexed = part.set_index("attribute")
        mean[model] = indexed.loc[attributes, "mean"].tolist()
        std[model] = indexed.loc[attributes, "std"].tolist()
    return mean, std


def seed_points(frame, metric, attributes=None):
    """per-seed の表から、category ごとの seed 値を手法別に集める。"""
    points = {}
    for model in MODEL_COLOR:
        part = frame[frame["model"] == model]
        if attributes is None:
            points[model] = [part[metric].tolist()]
            continue
        points[model] = [part[part["attribute"] == name][metric].tolist() for name in attributes]
    return points


def method_bars(axis, categories, mean, std, points, title):
    """category ごとに手法別の棒（平均±SD）を並べ、seed ごとの値を点で重ねる。"""
    positions = np.arange(len(categories))
    width = 0.8 / len(mean)
    offsets = np.linspace(-0.4 + width / 2, 0.4 - width / 2, len(mean))
    for offset, model in zip(offsets, mean, strict=True):
        centers = positions + offset
        error = {"ecolor": "#52514e", "elinewidth": 1.0, "capsize": 3}
        axis.bar(centers, mean[model], width, yerr=std[model], color=MODEL_COLOR[model], label=model, error_kw=error)
        for center, values in zip(centers, points[model], strict=True):
            axis.scatter(np.full(len(values), center), values, **SEED_MARK)
    axis.set_xticks(positions, categories)
    axis.set_title(title)
    axis.margins(y=0.20)


def method_dots(axis, categories, mean, std, points, title):
    """category ごとに手法別の点（平均±SD）を並べ、seed ごとの値を重ねる。

    0 起点の棒だと 0.004 前後の差が潰れて、節の主張が図に出ない。点は面積の意味を持たないので、
    軸を data 範囲に合わせても誤読にならない。
    """
    positions = np.arange(len(categories))
    offsets = np.linspace(-0.18, 0.18, len(mean))
    for offset, model in zip(offsets, mean, strict=True):
        centers = positions + offset
        style = {"color": MODEL_COLOR[model], "marker": "o", "markersize": 7, "linestyle": "none"}
        axis.errorbar(centers, mean[model], yerr=std[model], capsize=4, elinewidth=1.2, label=model, **style)
        for center, values in zip(centers, points[model], strict=True):
            axis.scatter(np.full(len(values), center), values, **SEED_MARK)
    axis.set_xticks(positions, categories)
    axis.set_xlim(-0.5, len(categories) - 0.5)
    axis.set_title(title)
    axis.margins(y=0.35)


def legend_on_top(figure, axis, title, note):
    """panel から凡例を取り出し、図の読み方を添えて figure の上に置く。"""
    handles, labels = axis.get_legend_handles_labels()
    figure.legend(handles, labels, loc="upper center", ncol=len(labels), bbox_to_anchor=(0.5, 1.10))
    figure.suptitle(f"{title}   ({note}, dots: each of {len(SEEDS)} seeds)", y=1.18, fontsize=12)

## 全体性能

まず「invariant 化でいくら払ったか」を見る。ここは属性を使わないので **test split 全行**が母集団になる。

`cross_entropy` も見る。AUROC が同じでも確率の較正が崩れていることがあるため。
こちらは**小さいほど良い**ので、差の符号が他の列と逆になる。

In [ ]:
columns = ["accuracy", "balanced_accuracy", "auroc", "cross_entropy"]
overall = performance_by_method.pivot(index="metric", columns="model", values=["mean", "std"])
overall = overall.loc[columns]
overall.columns = [f"{model} {statistic}" for statistic, model in overall.columns]
overall["Δ mean (invariant − ResNet)"] = overall[f"{INVARIANT} mean"] - overall[f"{RESNET} mean"]
overall.round(4)

per-seed の値も並べる。平均は 1 本の外れで動くので、3 点がどう散っているかを先に見る。

In [ ]:
performance.pivot(index="seed", columns="model", values=columns).round(4)

### 図で見る

差が 0.005 前後なので、0 起点の棒だと 4 枚とも同じ絵になり、節の主張が図に出ない。
点と SD のひげで描き、軸を data 範囲に合わせる。**点は面積の意味を持たない**ので、
0 を含まない軸でも棒のような誤読にならない。縦軸の幅が panel ごとに違う点に注意する。

In [ ]:
panels = [
    ("accuracy", "Accuracy"),
    ("balanced_accuracy", "Balanced accuracy"),
    ("auroc", "AUROC"),
    ("cross_entropy", "Cross entropy (lower is better)"),
]
figure, axes = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 3.6))
for axis, (metric, title) in zip(axes, panels, strict=True):
    mean, std = method_stats(performance_by_method, metric)
    points = seed_points(performance, metric)
    delta = mean[INVARIANT][0] - mean[RESNET][0]
    method_dots(axis, [""], mean, std, points, f"{title}\nΔ {delta:+.4f}")
legend_on_top(
    figure,
    axes[0],
    f"Overall performance ({SPLIT} split, n={performance['n_examples'].iloc[0]:,})",
    "marker: mean ± SD",
)
plt.show()

## 属性ごとの公平性

`Eopp1` は群間の TPR の最大差、`Eopp0` は TNR の最大差、`Eodds` はその平均。worst と best も
同じ表にあるので、gap の縮みが worst の改善なのか best の低下なのかを読める。

**`min n` と `min n positive` を先に見る。** gap は必ず一番外れた群が決めるので、その群が何人かを
知らないと差を読めない。これらは split が同じなので seed によらず一定になる。

**race は White / Asian / Black に絞ってある。** test split の race は 6 カテゴリあるが、残り 3 つは
n=3,280 / 356 / 49 で、陽性が 32 例・5 例しかない群の TPR が gap を決めてしまう。この絞り込みのため、
**race の値は学習側が記録した `val/race/*` とは別定義**になる。突き合わせない。
他の 3 属性は学習側と同じ定義のまま。

In [ ]:
metrics = ["Eopp1", "Eopp0", "Eodds", "AUROC gap", "bACC gap", "worst AUROC", "worst bACC"]
summary = fairness_by_method[fairness_by_method["metric"].isin(metrics)]
summary = summary.pivot(index=["attribute", "metric"], columns="model", values=["mean", "std"])
summary.columns = [f"{model} {statistic}" for statistic, model in summary.columns]
summary["Δ mean"] = summary[f"{INVARIANT} mean"] - summary[f"{RESNET} mean"]
support = fairness.groupby("attribute")[["min n", "min n positive"]].first()
summary.loc[ATTRIBUTES].round(4).join(support, on="attribute")

### gap を図で見る

属性をまたいで gap の大きさを比べる。x 軸に `min n` を添えて、どれだけの人数で測った差かを
同時に見る。ひげ（SD）が重なっている属性では、**平均の差を手法の差として読めない**。

In [ ]:
support = fairness.groupby("attribute")["min n"].first()
labels = [f"{name}\n(min n={int(support[name]):,})" for name in ATTRIBUTES]
panels = ["Eopp1", "Eopp0", "Eodds"]
figure, axes = plt.subplots(1, len(panels), figsize=(4.2 * len(panels), 3.8), sharey=True)
for axis, metric in zip(axes, panels, strict=True):
    mean, std = method_stats(fairness_by_method, metric, ATTRIBUTES)
    points = seed_points(fairness, metric, ATTRIBUTES)
    method_bars(axis, labels, mean, std, points, metric)
    axis.tick_params(axis="x", labelsize=8)
axes[0].set_ylabel("gap (lower is better)")
legend_on_top(figure, axes[0], f"Fairness gaps per attribute ({SPLIT} split)", "bar: mean ± SD")
plt.show()

### worst → best の幅

gap が縮んだのが **worst の改善**なのか **best の低下**なのかは、gap の値だけでは区別できない。
線が 3 seed の平均、点が seed ごとの値。線が短く、かつ右にあるのが望ましい。

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(7.8, 4.0), sharey=True)
offsets = np.linspace(0.18, -0.18, len(MODEL_COLOR))
for axis, metric in zip(axes, ["AUROC", "bACC"], strict=True):
    mean, _ = method_stats(fairness_by_method, f"worst {metric}", ATTRIBUTES)
    best, _ = method_stats(fairness_by_method, f"best {metric}", ATTRIBUTES)
    for offset, (model, color) in zip(offsets, MODEL_COLOR.items(), strict=True):
        for position, name in enumerate(ATTRIBUTES):
            span, height = [mean[model][position], best[model][position]], [position + offset] * 2
            axis.plot(span, height, color=color, linewidth=1.8, solid_capstyle="round", zorder=3)
            seeds = fairness[(fairness["model"] == model) & (fairness["attribute"] == name)]
            axis.scatter(seeds[f"worst {metric}"], [position + offset] * len(seeds), **SEED_MARK)
            axis.scatter(seeds[f"best {metric}"], [position + offset] * len(seeds), **SEED_MARK)
        axis.plot([], [], color=color, marker="o", label=model)
    axis.set_yticks(range(len(ATTRIBUTES)), ATTRIBUTES)
    axis.set_title(f"worst → best {metric}")
    axis.set_xlabel(metric)
axes[0].invert_yaxis()
legend_on_top(figure, axes[0], f"Group spread per attribute ({SPLIT} split)", "line: mean worst → mean best")
plt.show()

---

## 読み取り

まとめは [reports/overall_comparison.md](reports/overall_comparison.md) に置く。

**n=3 の SD は目安であって検定ではない。** 自由度 2 なので、SD 自体が大きく揺れる。
ここで言えるのは「ひげが重なるかどうか」「3 点が分離しているかどうか」までとする。

他に効いている条件:

- checkpoint は 6 本とも **val AUROC** で選んでいる。公平性で選んだ checkpoint ではない
- race は 3 カテゴリに絞ってある。落とした 3 カテゴリ（n=3,280 / 356 / 49）の患者は、
  この表のどの行にも入っていない
- ws11 で回した 6 本は `trainer.deterministic=False`。同じ seed でも完全再現はしない
- 交差群（age × sex × race）までは見ていない。見る場合は、同じ予測 cache から
  [iterative_probe](../iterative_probe/) の `groups.py` と同じ切り方で群を作る